## 1. Imports & Setup


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')


## 2. Load Dataset


In [16]:
DATA_PATH = '../global_bleaching_environmental.csv'

data = pd.read_csv(DATA_PATH, na_values=['nd', 'NA', ''])
print(f"Raw shape: {data.shape}")
print(f"\nFirst few rows:")

Raw shape: (41361, 62)

First few rows:


In [3]:
data.shape

(41361, 62)

In [17]:
drop_columns = ['Site_ID', 'Sample_ID', 'Data_Source','Ocean_Name', 'Reef_ID', 'Realm_Name',
       'Ecoregion_Name', 'Country_Name', 'State_Island_Province_Name',
       'City_Town_Name', 'Site_Name', 'Date_Day',
       'Substrate_Name', 'Site_Comments', 'Sample_Comments',
       'Bleaching_Comments', 'Bleaching_Level', 'Percent_Cover', 'SSTA_Mean', 'Exposure',
       ] # Exposure - interesting

data.drop(columns= drop_columns, inplace= True)


In [18]:
data = data[data['SSTA'] > 0]

data = data[data['Date_Year'] >= 2003]

PAPER_FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
TARGET = 'Percent_Bleaching'
data = data.dropna(subset=['Percent_Bleaching'])


In [19]:
data.shape

(19727, 42)

In [14]:
print(data[TARGET].isna().any().any())
print(data[PAPER_FEATURES].isna().any().any())

False
True


## 6. Train/Test Split


In [7]:
from sklearn.impute import SimpleImputer

# Train/test split ก่อน
X = data[PAPER_FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Impute (fit เฉพาะ train, transform ทั้งคู่)
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

## 7. LazyPredict Model Comparison (Optional)


In [8]:
try:
    from lazypredict.Supervised import LazyRegressor

    lazy_regressor = LazyRegressor(
        verbose=0,
        ignore_warnings=True,
        custom_metric=None
    )

    lazy_models, lazy_predictions = lazy_regressor.fit(
        X_train, X_test, y_train, y_test
    )

    print("LazyPredict model comparison (Top 15):")
    display(lazy_models.head(15))

except Exception as exc:
    lazy_models = None
    print("LazyPredict skipped.")
    print("ถ้าต้องการรัน cell นี้ ให้ติดตั้งก่อนด้วย: %pip install lazypredict")
    print(f"Reason: {type(exc).__name__}: {exc}")


LazyPredict model comparison (Top 15):


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,0.692795,0.693185,10.528536,1.351640
RandomForestRegressor,0.666090,0.666513,10.976626,3.614099
BaggingRegressor,0.632917,0.633382,11.508967,0.372803
XGBRegressor,0.598979,0.599488,12.029221,0.447826
ExtraTreeRegressor,0.486212,0.486863,13.615897,0.024306
HistGradientBoostingRegressor,0.484510,0.485163,13.638433,0.146492
DecisionTreeRegressor,0.461714,0.462396,13.936722,0.056785
KNeighborsRegressor,0.316636,0.317502,15.702916,0.045382
GradientBoostingRegressor,0.314383,0.315252,15.728775,1.113667


# Optuna

In [9]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor

In [10]:
def et_objective(trial):

    model = ExtraTreesRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

et_study = optuna.create_study(direction="maximize")

et_study.optimize(et_objective, n_trials=50)

print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-06 10:25:04,944] A new study created in memory with name: no-name-4378cddd-3375-4a65-a869-03f82bba9123
[I 2026-05-06 10:25:07,089] Trial 0 finished with value: 0.48648181121816136 and parameters: {'n_estimators': 165, 'max_depth': 21, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.48648181121816136.
[I 2026-05-06 10:25:09,074] Trial 1 finished with value: 0.5622153926073111 and parameters: {'n_estimators': 268, 'max_depth': 36, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.5622153926073111.
[I 2026-05-06 10:25:10,857] Trial 2 finished with value: 0.5607693493679051 and parameters: {'n_estimators': 215, 'max_depth': 28, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.5622153926073111.
[I 2026-05-06 10:25:11,292] Trial 3 finished with value: 0.3592394982859167 and parameters: {'n_estimators': 203, 'max_depth': 13, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 1 with val

========== ET BEST ==========
{'n_estimators': 137, 'max_depth': 26, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6197


## 8. USE Baseline

In [11]:
et = ExtraTreesRegressor(
    n_estimators=137,
    max_depth=26,
    min_samples_leaf=1,
    max_features='log2',
    criterion='squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

et.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 9. Evaluate Model

In [12]:
y_pred = et.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("=" * 50)
print("BASELINE RESULTS (single seed = 42)")
print("=" * 50)
print(f"R²    = {r2:.4f}    (paper: 0.25)")
print(f"RMSE  = {rmse:.4f}    (paper: 7.91)")
print(f"MAE   = {mae:.4f}")
print(f"Accuracy (100 - MAE) = {100 - mae:.2f}%   (paper: 96%)")
print(f"\n⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง")

BASELINE RESULTS (single seed = 42)
R²    = 0.6777    (paper: 0.25)
RMSE  = 10.7907    (paper: 7.91)
MAE   = 4.5471
Accuracy (100 - MAE) = 95.45%   (paper: 96%)

⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง


## 10. K-Fold Cross Validation (k=4)


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

cv = KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

# ✅ Wrap ใน Pipeline เพื่อให้ imputer fit ใหม่ทุก fold
et_cv = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', ExtraTreesRegressor(
        n_estimators=137,           
        max_depth=26,
        min_samples_leaf=1,
        max_features='log2',
        criterion='squared_error',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

cv_results = cross_validate(
    et_cv,
    X,
    y,
    cv=cv,
    scoring={
        'r2': 'r2',
        'neg_mse': 'neg_mean_squared_error',
        'neg_mae': 'neg_mean_absolute_error'
    },
    n_jobs=-1,
    return_train_score=False
)


kfold_df = pd.DataFrame({
    'fold': range(1, 5),
    'R²': cv_results['test_r2'],
    'RMSE': np.sqrt(-cv_results['test_neg_mse']),
    'MAE': -cv_results['test_neg_mae']
})
kfold_df['Accuracy'] = 100 - kfold_df['MAE']

print("K-Fold Cross Validation (k=4):")
print(kfold_df.to_string(index=False))
print()
print(f"Mean R²:       {kfold_df['R²'].mean():.4f} ± {kfold_df['R²'].std():.4f}")
print(f"Mean RMSE:     {kfold_df['RMSE'].mean():.4f} ± {kfold_df['RMSE'].std():.4f}")
print(f"Mean MAE:      {kfold_df['MAE'].mean():.4f} ± {kfold_df['MAE'].std():.4f}")
print(f"Mean Accuracy: {kfold_df['Accuracy'].mean():.2f}% ± {kfold_df['Accuracy'].std():.2f}%")

K-Fold Cross Validation (k=4):
 fold       R²      RMSE      MAE  Accuracy
    1 0.662141 11.314761 4.859468 95.140532
    2 0.649835 11.640458 4.992863 95.007137
    3 0.670748 11.301190 4.841860 95.158140
    4 0.647938 11.796827 5.024293 94.975707

Mean R²:       0.6577 ± 0.0108
Mean RMSE:     11.5133 ± 0.2456
Mean MAE:      4.9296 ± 0.0924
Mean Accuracy: 95.07% ± 0.09%
